[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benchuangxd/CSC3109-T16-Project/blob/main/notebooks/01_eda.ipynb)

# Exploratory Data Analysis

**CSC3109 - Machine Learning | Team 16**

This notebook performs an exploratory analysis of the image dataset before model training. It covers:

1. Dataset structure & class distribution
2. Sample image visualisation
3. Image dimension verification
4. Per-channel pixel statistics (mean & std for normalisation)
5. Pixel intensity distributions per class
6. Brightness & contrast analysis

In [ ]:
# ── Google Colab Setup ──────────────────────────────────────────────────────
# Run this cell first when using Google Colab. No effect when running locally.
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    from pathlib import Path

    REPO_URL  = "https://github.com/benchuangxd/CSC3109-T16-Project.git"
    REPO_PATH = Path("/content/CSC3109-T16-Project")

    if not REPO_PATH.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_PATH)], check=True)
    else:
        print(f"Repo already exists at {REPO_PATH}")

    %cd /content/CSC3109-T16-Project
    %pip install -q -r requirements.txt
    print("Colab setup complete.")
else:
    print("Running locally — skipping Colab setup.")

## 1. Imports & Setup

In [5]:
import os
import sys
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA_DIR    = ROOT / 'data' / 'set 16'    # Full dataset — split 70/30 into train/val
TEST_DIR    = ROOT / 'data' / 'val 16'    # Held-out test set
CLASSES     = ['beach', 'ferry_terminal', 'harbor', 'river']
TRAIN_RATIO = 0.7

print(f'Project root : {ROOT}')
print(f'Data dir     : {DATA_DIR}  (exists={DATA_DIR.exists()})')
print(f'Test dir     : {TEST_DIR}  (exists={TEST_DIR.exists()})')
print(f'Train/Val split : {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}')

## 2. Dataset Structure

In [ ]:
rows = []
for cls in CLASSES:
    data_imgs = list((DATA_DIR / cls).glob('*.*'))
    test_imgs = list((TEST_DIR / cls).glob('*.*'))
    n = len(data_imgs)
    n_train = int(TRAIN_RATIO * n)
    n_val   = n - n_train
    rows.append({
        'Class':     cls,
        'Dataset':   n,
        'Train (70%)': n_train,
        'Val (30%)':   n_val,
        'Test':      len(test_imgs),
    })

df_counts = pd.DataFrame(rows)
totals = pd.DataFrame([{
    'Class':     'TOTAL',
    'Dataset':   df_counts['Dataset'].sum(),
    'Train (70%)': df_counts['Train (70%)'].sum(),
    'Val (30%)':   df_counts['Val (30%)'].sum(),
    'Test':      df_counts['Test'].sum(),
}])
display(pd.concat([df_counts, totals], ignore_index=True))

## 3. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = np.arange(len(CLASSES))
w = 0.25
axes[0].bar(x - w, df_counts['Train (70%)'], w, label='Train (70%)', color='steelblue')
axes[0].bar(x,     df_counts['Val (30%)'],   w, label='Val (30%)',   color='coral')
axes[0].bar(x + w, df_counts['Test'],        w, label='Test',        color='seagreen')
axes[0].set_xticks(x)
axes[0].set_xticklabels(CLASSES, rotation=15)
axes[0].set_ylabel('Image Count')
axes[0].set_title('Train / Val / Test Split per Class')
axes[0].legend()

axes[1].pie(
    [df_counts['Train (70%)'].sum(), df_counts['Val (30%)'].sum(), df_counts['Test'].sum()],
    labels=['Train (70%)', 'Val (30%)', 'Test'],
    autopct='%1.1f%%',
    startangle=90,
    colors=['steelblue', 'coral', 'seagreen'],
)
axes[1].set_title('Overall Split Distribution')

plt.suptitle('Class & Split Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Sample Images per Class

In [ ]:
N_SAMPLES = 5

fig, axes = plt.subplots(len(CLASSES), N_SAMPLES, figsize=(N_SAMPLES * 3, len(CLASSES) * 3))
fig.suptitle('Sample Images - Dataset (set 16)', fontsize=14)

for row, cls in enumerate(CLASSES):
    img_paths = list((DATA_DIR / cls).glob('*.*'))
    samples   = random.sample(img_paths, min(N_SAMPLES, len(img_paths)))
    for col, path in enumerate(samples):
        img = Image.open(path).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(
                cls.replace('_', ' ').title(), loc='left', fontsize=10, pad=3
            )

plt.tight_layout()
plt.show()

## 5. Image Dimension Verification

In [ ]:
size_counts = defaultdict(int)

for cls in CLASSES:
    for path in (DATA_DIR / cls).glob('*.*'):
        try:
            with Image.open(path) as img:
                size_counts[img.size] += 1
        except Exception:
            pass

print('Unique (width x height) sizes in dataset:\n')
for (w, h), count in sorted(size_counts.items(), key=lambda x: -x[1]):
    pct = count / sum(size_counts.values()) * 100
    print(f'  {w} x {h} px  --  {count:,} images  ({pct:.1f}%)')

dominant = max(size_counts, key=size_counts.get)
print(f'\nDominant size: {dominant[0]} x {dominant[1]} px')
print('Config IMAGE_SIZE=224 resizes/crops these to 224x224 during transforms.')

## 6. Per-Channel Pixel Statistics

Computing dataset-wide mean and std per RGB channel across the full dataset (`set 16`).
These values should be used in `transforms.Normalize()` for **all** model notebooks.

In [ ]:
print('Computing per-channel statistics across the full dataset (set 16)...\n')

ch_sum    = np.zeros(3, dtype=np.float64)
ch_sq_sum = np.zeros(3, dtype=np.float64)
n_pixels  = 0

for cls in CLASSES:
    for path in (DATA_DIR / cls).glob('*.*'):
        try:
            arr    = np.array(Image.open(path).convert('RGB'), dtype=np.float32) / 255.0
            pixels = arr.reshape(-1, 3)
            ch_sum    += pixels.sum(axis=0)
            ch_sq_sum += (pixels ** 2).sum(axis=0)
            n_pixels  += len(pixels)
        except Exception:
            pass

mean = ch_sum / n_pixels
std  = np.sqrt(ch_sq_sum / n_pixels - mean ** 2)

print(f'{"Channel":<10} {"Mean":>8} {"Std":>8}')
print('-' * 30)
for i, name in enumerate(['Red', 'Green', 'Blue']):
    print(f'{name:<10} {mean[i]:>8.4f} {std[i]:>8.4f}')

print(f'\nmean = [{mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f}]')
print(f'std  = [{std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f}]')
print('\nUse these in transforms.Normalize(mean=[...], std=[...]) for all model notebooks.')

## 7. Pixel Intensity Distribution per Class

In [ ]:
SAMPLE_N  = 60
CH_COLORS = ['tomato', 'mediumseagreen', 'cornflowerblue']
CH_NAMES  = ['Red', 'Green', 'Blue']

fig, axes = plt.subplots(
    len(CLASSES), 3,
    figsize=(14, 3.5 * len(CLASSES)),
    sharey='row',
)
fig.suptitle('Pixel Intensity Distribution - Dataset (sampled)', fontsize=13, y=1.01)

for row, cls in enumerate(CLASSES):
    paths   = list((DATA_DIR / cls).glob('*.*'))
    samples = random.sample(paths, min(SAMPLE_N, len(paths)))
    arrays  = []
    for p in samples:
        try:
            arrays.append(
                np.array(Image.open(p).convert('RGB'), dtype=np.float32) / 255.0
            )
        except Exception:
            pass
    pixels = np.concatenate([a.reshape(-1, 3) for a in arrays], axis=0)

    for ch in range(3):
        axes[row, ch].hist(pixels[:, ch], bins=64, color=CH_COLORS[ch], alpha=0.75, density=True)
        axes[row, ch].set_xlim(0, 1)
        if row == 0:
            axes[row, ch].set_title(CH_NAMES[ch], fontsize=11)
        if ch == 0:
            axes[row, ch].set_ylabel(cls.replace('_', '\n'), fontsize=9)

plt.tight_layout()
plt.show()

## 8. Brightness & Contrast Distribution per Class

In [ ]:
SAMPLE_N = 100

brightness = {cls: [] for cls in CLASSES}
contrast   = {cls: [] for cls in CLASSES}

for cls in CLASSES:
    paths   = list((DATA_DIR / cls).glob('*.*'))
    samples = random.sample(paths, min(SAMPLE_N, len(paths)))
    for p in samples:
        try:
            arr = np.array(Image.open(p).convert('L'), dtype=np.float32) / 255.0
            brightness[cls].append(arr.mean())
            contrast[cls].append(arr.std())
        except Exception:
            pass

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette = sns.color_palette('Set2', len(CLASSES))

for ax, data, title, ylabel in [
    (axes[0], brightness, 'Mean Brightness per Class',   'Mean Pixel Value (grayscale)'),
    (axes[1], contrast,   'Contrast (Std Dev) per Class', 'Std Dev of Pixel Values'),
]:
    parts = ax.violinplot(
        [data[c] for c in CLASSES],
        positions=range(1, len(CLASSES) + 1),
        showmeans=True,
        showmedians=False,
    )
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(palette[i])
        pc.set_alpha(0.75)
    ax.set_xticks(range(1, len(CLASSES) + 1))
    ax.set_xticklabels([c.replace('_', '\n') for c in CLASSES], fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

plt.suptitle('Brightness & Contrast Distribution', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Summary

| Item | Value |
|---|---|
| Classes | beach, ferry_terminal, harbor, river |
| Full dataset (`set 16`) | 2,800 (700 per class - perfectly balanced) |
| Train split (70%) | 1,960 (490 per class) |
| Val split (30%) | 840 (210 per class) |
| Test set (`val 16`) | 400 (100 per class) |
| Source image size | 256 x 256 px |
| Model input size | 224 x 224 px (resize + crop in transforms) |
| Class imbalance | None - no oversampling needed |

**Normalisation values** computed in Section 6 should be used in all model notebooks:

```python
transforms.Normalize(mean=[R, G, B], std=[R, G, B])
```

**Key observations:**
- Dataset is perfectly balanced across all classes.
- All images are consistently 256x256 px; preprocessing is straightforward.
- Classes show distinct brightness and colour profiles, indicating strong visual discriminability.
- `set 16` is split 70/30 for train/val; `val 16` serves as a held-out test set.